# Week 3


Today's lecture has two main focuses:

__Part 1: Wrapping up Data Collection:__ We're finishing our deep dive into data collection using the OpenAlex API. 
We'll retrieve:      

- information about the author, such as country and citation count.    
- information about publications, such as titles and abstracts.     

The goal is to become comfortable handling **large, real-world datasets.**

 *A little warning*: In this week's exercises you will retreive and process a **large amount of data**. If you find the process tedious and overwhelming at times, you are not alone! That's a completely normal part of dealing with real data.     
 
 But don't worry, I'll guide you through step by step, and I'm always available to help, on Teams or in class. There are three simple rules for success:    

- Follow thoroughly the step-by-step instructions.   
- Stay on top of your work weekly.    
- If you are stuck, come and talk to me during class/ask me via Teams.    

__Parts 2 and 3: Introduction to Data Visualization + Visualizing Distributions:__ Next, we shift to a lighter topic: **Data Visualization.** There will be some video lectures on general aspects of Data Visualization. Then we'll start with visualizing histograms. You may feel already confident with plotting histograms, but we'll explore new angles, especially when it comes to visualizing data that's spread out across a wide range. You will start seeing the value of the data we've collected and starting to analyze it in practical ways.


This is probably the most intense lecture of the course in terms of coding. Once you are on the other side from this one, it will feel much smoother from there. 

## Part 1: Wrapping up the data collection

You're now equipped to tackle more challenging tasks with the OpenAlex API. In this part of the course, we will build the **Computational Social Science dataset step by step**. 

The key idea is that each dataset builds on the previous one. Think of this as a pipeline:    
- **Stage 1:** We start from a list of *IC2S2 authors* (from Week 2).    
- **Stage 2:** We collect their scientific papers.    
- **Stage 3:** From those papers, we identify their *co-authors* — researchers who wrote papers together with them — and then collect papers written by those co-authors.    
- **Stage 4:** Finally, we merge everything into one unified dataset representing the Computational Social Science field.    


**Why do we collect data this way?**
Researchers do not attend all conferences, even if those conferences are relevant to their work. If we only included IC2S2 participants, we would capture only a visible subset of the field. For this reason, we expand our scope to include the collaborators of the IC2S2 authors — the *IC2S2 co-authors*. By moving one step outward through co-authorship links, we obtain a more realistic approximation of the Computational Social Science research community. In network science, this strategy is closely related to [snowball sampling](https://en.wikipedia.org/wiki/Snowball_sampling): starting from a seed set and expanding through observed connections.

**In Exercise 1**, you will collect all research articles written by the IC2S2 authors (~1500 researchers). Even though this seems manageable, efficiency already matters — the code you write here will be reused in the next exercise.

**In Exercise 2**, we expand to include the collaborators of the IC2S2 authors (the IC2S2 co-authors). This step significantly increases the dataset size. You'll be managing **many** API requests here. This makes it crucial to write code that's not just functional but also efficient.

**Overview of the Datasets we collect** 
![Data collection pipeline](https://raw.githubusercontent.com/lalessan/comsocsci2026/master/images/schema_data.png)


<div style="
  border-left: 6px solid #c0392b;
  background-color: #f8d7da;
  padding: 1em 1.2em;
  border-radius: 6px;
  max-width: 650px;
  margin: 1.5em auto;
  color: #222;
">

**⚠ Before Exercise 1: Get an API key**

Starting **February 13, 2026**, an API key is required to use the OpenAlex API.

Steps for success: 

- [Sign up](https://openalex.org/login?redirect=/settings/api) and authenticate.     
- [Copy your API key](https://openalex.org/settings/api).   
- add ``api_key=YOUR_KEY`` to your API calls. When you use the ``requests`` library you can add it to your ``params`` dictionary.    


</div>




> **Exercise 1: Collecting Research Articles from IC2S2 Authors**
>
>In this exercise, we'll leverage the OpenAlex API to gather information on research articles authored by participants of the IC2S2 2025 conference, referred to as *IC2S2 authors*. **Before you start, please ensure you read through the entire exercise.**
>
> 
> **Steps:**
>  
> 1. **Retrieve Data:** Start with the dataset of *IC2S2 authors* you collected in Week 2, Exercise 3 (called dataset D1 in the figure above). Use the OpenAlex API [works endpoint](https://docs.openalex.org/api-entities/works) to fetch their research articles. For each article, retrieve the following details:
>    - _id_: The unique OpenAlex ID for the work.
>    - _publication_year_: The year the work was published.
>    - _cited_by_count_: The number of times the work has been cited by other works.
>    - _author_ids_: The OpenAlex IDs for the authors of the work.
>    - _title_: The title of the work.
>    - _abstract_inverted_index_: The abstract of the work, formatted as an inverted index.
> 
>     **Important Note on Paging:** By default, the OpenAlex API limits responses to 25 works per request. For more efficient data retrieval, I suggest to adjust this limit to 200 works per request. Even with this adjustment, you will need to implement pagination to access all available works for a given query. This ensures you can systematically retrieve the complete set of works beyond the initial 200. Find guidance on implementing pagination [here](https://docs.openalex.org/how-to-use-the-api/get-lists-of-entities/paging#cursor-paging).
>
> 2. **Data Storage:** Organize the retrieved information into two Pandas DataFrames and save them to two files in a suitable format:
>    - Dataset D2: The *IC2S2 papers* dataset should include: *id, publication\_year, cited\_by\_count, author\_ids*.
>    - Dataset D3: The *IC2S2 abstracts* dataset should include: *id, title, abstract\_inverted\_index*.
>  
>
> **Filters:**
> To ensure the data we collect is relevant and manageable, apply the following filters:
>     
>    - Only include *IC2S2 authors* with a total work count between 5 and 5,000.    
>    - Retrieve only works that have received more than 10 citations.    
>    - Limit to works authored by fewer than 10 individuals.    
>    - Include only works relevant to Computational Social Science (focusing on: Sociology OR Psychology OR Economics OR Political Science) AND intersecting with a quantitative discipline (Mathematics OR Physics OR Computer Science), as defined by their [Concepts](https://docs.openalex.org/api-entities/works/work-object#concepts). *Note*: here we only consider Concepts at *level=0* (the most coarse definition of concepts).     
>
> **Efficiency Tips:**
> Writing efficient code in this exercise is **crucial**. To speed up your process:
> 
> - **Apply filters directly in your request:** When possible, use the [filter parameter](https://docs.openalex.org/api-entities/works/filter-works) of the *works* endpoint to apply the filters above directly in your API request, ensuring only relevant data is returned. Learn about combining multiple filters [here](https://docs.openalex.org/how-to-use-the-api/get-lists-of-entities/filter-entity-lists).  
> - **Bulk requests:** Instead of sending one request for each author, you can use the [filter parameter](https://docs.openalex.org/api-entities/works/filter-works) to query works by multiple authors in a single request. *Note: My testing suggests that can only include up to 25 authors per request.*
> - **Use multiprocessing:** Implement multiprocessing to handle multiple requests simultaneously. I highly recommmend [Joblib’s Parallel](https://joblib.readthedocs.io/en/stable/) function for that, and [tqdm](https://tqdm.github.io/) can help monitor progress of your jobs. Remember to stay within [the rate limit](https://docs.openalex.org/how-to-use-the-api/rate-limits-and-authentication) of 100 requests per second.
>
>
>   
> For reference, employing these strategies allowed me to fetch the data in about 30 seconds using 5 cores on my laptop. I obtained a dataset of approximately 25 MB (including both the *IC2S2 abstracts* and *IC2S2 papers* files).
> 
>
> **Data Overview and Reflection questions:** Answer the following questions: 
> 
> - **Dataset summary.** How many works are listed in your Dataset D2 (*IC2S2 papers*) dataframe? How many unique researchers have co-authored these works?     
> - **Efficiency in code.** Describe the strategies you implemented to make your code more efficient. How did your approach affect your code's execution time?    
> - **Filtering Criteria and Dataset Relevance** Reflect on the rationale behind setting specific thresholds for the total number of works by an author, the citation count, the number of authors per work, and the relevance of works to specific fields. How do these filtering criteria contribute to the relevance of the dataset you compiled? Do you believe any aspects of Computational Social Science research might be underrepresented or overrepresented as a result of these choices?    

In [ ]:
from joblib import Parallel, delayed
import requests
import pandas as pd
from tqdm.notebook import tqdm
import numpy as np

In [2]:
D1 = pd.read_csv(r"D1.csv")

# Filter
# Only include *IC2S2 authors* with a total work count between 5 and 5,000
D1 = D1[(D1["works_count"] >= 5) & (D1["works_count"] <= 5000)].reset_index(drop=True)
D1

,Name,id,display_name,works_api_url,h_index,works_count,country_code
0,marc keuschnigg,https://openalex.org/A5058700879,Marc Keuschnigg,https://api.openalex.org/works?filter=author.i...,17.0,76.0,NaN
1,sonia yeh,https://openalex.org/A5029945844,Sonia Yeh,https://api.openalex.org/works?filter=author.i...,41.0,224.0,GB
2,peter hedström,https://openalex.org/A5071769816,Peter Hedström,https://api.openalex.org/works?filter=author.i...,32.0,92.0,SE
3,nina tahmasebi,https://openalex.org/A5003859694,Nina Tahmasebi,https://api.openalex.org/works?filter=author.i...,15.0,110.0,SE
4,hendrik erz,https://openalex.org/A5020654056,Hendrik Erz,https://api.openalex.org/works?filter=author.i...,2.0,38.0,NaN
...,...,...,...,...,...,...,...
1341,carl nordlund,https://openalex.org/A5033663242,Carl Nordlund,https://api.openalex.org/works?filter=author.i...,6.0,24.0,NaN
1342,christian steglich,https://openalex.org/A5030685996,Christian Steglich,https://api.openalex.org/works?filter=author.i...,42.0,104.0,NaN
1343,neha gondal,https://openalex.org/A5034089559,Neha Gondal,https://api.openalex.org/works?filter=author.i...,11.0,24.0,US
1344,victor møller poulsen,https://openalex.org/A5059830287,Victor Møller Poulsen,https://api.openalex.org/works?filter=author.i...,2.0,7.0,US


In [3]:
# Get author_ID from OpenAlex_ID
def get_id(openalex_id):
    return openalex_id.split("/")[-1]
original_authors_id = D1["id"].apply(get_id)

# Do author ID chunking
def chunked(authors, size):
    for i in range(0, len(authors), size):
        yield authors[i:i + size]

author_batches = list(chunked(original_authors_id, 25))

# Url building for chunking
def build_works_url(author_batch):
    author_filter = "|".join(author_batch)
    return (
        "https://api.openalex.org/works"
        f"?filter=authorships.author.id:{author_filter}"
    )

In [ ]:
# Filter
# Include only works relevant to Computational Social Science 
# (focusing on: Sociology OR Psychology OR Economics OR Political Science) 
# AND intersecting with a quantitative discipline (Mathematics OR Physics OR Computer Science),
# as defined by their [Concepts]
SOCIAL_CONCEPTS = {"Sociology", "Psychology", "Economics", "Political science"}
QUANT_CONCEPTS = {"Mathematics", "Physics", "Computer science"}

def is_relevant_concepts(article):
    concepts = article["concepts"]
    social = any(c["level"] == 0 and c["display_name"] in SOCIAL_CONCEPTS for c in concepts)
    quant = any(c["level"] == 0 and c["display_name"] in QUANT_CONCEPTS for c in concepts)
    return social and quant

def get_response(works_api_url):   
    rows = []
    params = {"per_page": 200,
            "api_key": "MiOvuHcHHXnRrjyyjzjKba",
            "cursor":"*"}
    
    session = requests.Session()
    session.headers.update({"User-Agent": "research-script/1.0"})

    while True:
        response = session.get(works_api_url, params=params, timeout=30)
        result = response.json()
        results = result["results"]

        for article in results:
            # Concepts filter
            if not is_relevant_concepts(article): 
                continue

            authors = [authorship["author"]["id"] for authorship in article["authorships"]]
                
            # Filter
            # Retrieve only works that have received more than 10 citations.    
            # Limit to works authored by fewer than 10 individuals.
            if article["cited_by_count"] <=10 or len(authors)>=10: 
                continue

            rows.append({
                "id": article["id"],
                "publication_year": article["publication_year"],
                "cited_by_count": article["cited_by_count"],
                "author_ids": [author.split("/")[-1] for author in authors if author is not None],
                "title": article["title"],
                "abstract_inverted_index": article["abstract_inverted_index"]
            })

        next_cursor = result["meta"]["next_cursor"]
        if not next_cursor:
            break
        params["cursor"] = next_cursor
    
    return pd.DataFrame(rows)

# Parallel fetching with tqdm
urls = [build_works_url(batch) for batch in author_batches]

# Multiprocessing
metadata_list = Parallel(n_jobs=4)(
    delayed(get_response)(url)
    for url in tqdm(urls, desc="Fetching works (batched)")
)

# Filter out None results and combine
metadata_list = [df for df in metadata_list if df is not None and not df.empty]
metadata_df = pd.concat(metadata_list, ignore_index=True)

D2 = metadata_df[["id", "publication_year", "cited_by_count", "author_ids"]].copy()
D3 = metadata_df[["id", "title", "abstract_inverted_index"]].copy()

Fetching works (batched):   0%|          | 0/54 [00:00<?, ?it/s]

In [ ]:
metadata_df = metadata_df.drop_duplicates(subset=["id"])
D2 = D2.drop_duplicates(subset=["id"])
D3 = D3.drop_duplicates(subset=["id"])

In [ ]:
# metadata_df.to_csv("metadata_df.csv", index=False)
# D2.to_csv("D2.csv", index=False)
# D3.to_csv("D3.csv", index=False)

# D2 = pd.read_csv(r"D2.csv")
# D3 = pd.read_csv(r"D3.csv")

In [ ]:
metadata_df

In [7]:
D2

,id,publication_year,cited_by_count,author_ids
0,https://openalex.org/W2152796930,2010.0,1548,"['A5071769816', 'A5029640601']"
1,https://openalex.org/W2088900896,2000.0,1436,"['A5100661890', 'A5108538229', 'A5112454407', ..."
2,https://openalex.org/W1989597713,2009.0,1168,"['A5071165387', 'A5037969281']"
3,https://openalex.org/W2103177132,1999.0,1097,['A5029945844']
4,https://openalex.org/W4245013494,1999.0,1092,['A5029945844']
...,...,...,...,...
15218,https://openalex.org/W2950728552,2019.0,19,"['A5088239541', 'A5100722234', 'A5044145078', ..."
15219,https://openalex.org/W3103732528,2019.0,14,"['A5100572173', 'A5043029070', 'A5039315344', ..."
15220,https://openalex.org/W2617544931,1997.0,13,"['A5080030270', 'A5019212344', 'A5111454621']"
15221,https://openalex.org/W2990181595,2019.0,13,"['A5100572173', 'A5043029070', 'A5039315344', ..."


In [8]:
D3

,id,title,abstract_inverted_index
0,https://openalex.org/W2152796930,Causal Mechanisms in the Social Sciences,"{'During': [0], 'the': [1, 14, 21, 29, 38, 45,..."
1,https://openalex.org/W2088900896,A new LDA-based face recognition system which ...,NaN
2,https://openalex.org/W1989597713,Effects of Questionnaire Length on Participati...,"{'This': [0], 'paper': [1], 'investigates': [2..."
3,https://openalex.org/W2103177132,NaN,NaN
4,https://openalex.org/W4245013494,Response Rate in Academic Studies-A Comparativ...,"{'A': [0], 'study': [1, 148], 'was': [2, 72], ..."
...,...,...,...
15218,https://openalex.org/W2950728552,Unsupervised Learning of Object Structure and ...,"{'Extracting': [0], 'and': [1, 5, 28, 44, 60, ..."
15219,https://openalex.org/W3103732528,Memory Based Trajectory-conditioned Policies f...,"{'Reinforcement': [0], 'learning': [1, 133], '..."
15220,https://openalex.org/W2617544931,Dynamic Model and Causal Knowledge-Based Fault...,NaN
15221,https://openalex.org/W2990181595,Self-Imitation Learning via Trajectory-Conditi...,"{'Imitation': [0], 'learning': [1, 15, 39, 101..."


<div style="
  border:1px solid #5b2ca0;
  border-radius:10px;
  margin:20px 0;
  background:#ede0ff;
  padding:22px;
  color:#2b003d;
  line-height:1.6;
  max-width:100%;
  box-sizing:border-box;
  overflow-wrap:break-word;
">
<p>
To maximise learning and ensure a correct solution, avoid LLMs to solve the exercise. 
As ususal it is important to solve the exercise in a "modular" way.
Start by making an API request for a single author, inspect the response carefully, and try a few different names to discover potential issues and address them.  
Once you’re satisfied with your approach, work on scaling up the solution and run the requests for many authors. 
If you'd like, once you have a working code, you can ask an LLM for suggestions on how to improve efficiency.
</p>
</div>